# RFP 리스크 분석 — 프로젝트 개요

공공 AI·IT 구축 RFP 10개에서 요구사항을 추출하고, 제안 견적과 계약 검토가 필요한 조항을 분류하는 연구 프로젝트입니다.

이 노트북은 **지금 무엇이 어디에 있는지**를 한눈에 보여줍니다. 분석은 하지 않습니다.

## 파이프라인 순서

```
RFP 원본(PDF·HWP)  →  Markdown 변환  →  요구사항 추출  →  LLM 라벨링  →  ML 비교
   RFP_data/          RFP_data/md/    data/processed/   reports/current/   (예정)
                                                         claude_runs/
```

## 주 라벨 (가격 산정 가능성)

| 라벨 | 의미 |
|---|---|
| `통상수용` | 추가 원가 없이 기본 수행팀 공수에 포함 |
| `견적반영` | 원가는 붙지만 원문 정보로 계산 가능 |
| `계약·질의검토` | blocker가 있어 계산 불가 |

보조 축 4개(`blockers`, `cost_basis`, `domain_dependency`, `build_difficulty`)를 함께 산출합니다.
자세한 내용은 `notebooks/02_labeling_experiment.ipynb`를 보세요.

## 관련 문서

- 연구 설계 전반: `docs/PROJECT_DIRECTION.md`
- 의사결정 이력: `docs/history/decisions-0*.md`
- 실험 결과: `reports/current/labeling_experiment_v0.1.0.md`

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

# 디렉터리별 파일 수. 어디에 산출물이 쌓여 있는지 감을 잡기 위한 것입니다.
DIRECTORY_ROLES = {
    'RFP_data': '원본 PDF·HWP와 분석용 Markdown',
    'data': '생성 데이터셋, 표본, 앵커 풀, 사람 검수 자료',
    'scripts': '추출·전처리·라벨링·검색 로직 (노트북이 아니라 여기가 기준)',
    'notebooks': '사용법 예제와 탐색적 분석',
    'reports': '보고서와 LLM 실행 결과',
    'docs': '연구 설계와 의사결정 기록',
    'tests': '단위·구조 검증',
}

for name, role in DIRECTORY_ROLES.items():
    path = ROOT / name
    count = sum(1 for item in path.rglob('*') if item.is_file()) if path.exists() else 0
    print(f'{name:<12} {count:>5} files   {role}')

In [ ]:
# 핵심 모듈. 역할별로 묶어서 어떤 파일을 읽어야 하는지 보여줍니다.
MODULES = {
    '데이터 추출·전처리': [
        'scripts/data/build_dataset.py',        # RFP Markdown -> requirements JSONL
        'scripts/data/preprocess_text.py',
        'scripts/data/eda_requirements.py',
        'scripts/data/sample_pilot.py',         # 파일럿 표본 추출
    ],
    '라벨 정의': [
        'scripts/labeling/label_schema.py',     # LabelResult, derive_primary_action
        'scripts/labeling/validate_label_schema.py',
    ],
    '앵커 (few-shot)': [
        'scripts/labeling/anchor_pool.py',      # 감사 게이트: 검토완료 앵커만 통과
        'scripts/labeling/anchor_retriever.py', # TF-IDF 층화 인출
    ],
    'LLM 실행': [
        'scripts/labeling/claude_client.py',    # 시스템 프롬프트와 API 어댑터
        'scripts/labeling/run_claude_labeling.py',  # 동기 실행
        'scripts/labeling/run_claude_batch.py',     # 배치 실행 (50% 저렴)
        'scripts/labeling/llm_token_tracker.py',    # 토큰·비용 추적
    ],
}

for group, paths in MODULES.items():
    print(f'\n[{group}]')
    for rel in paths:
        mark = 'O' if (ROOT / rel).exists() else 'X 없음'
        print(f'  {mark:<6} {rel}')

In [ ]:
import json

# 현재 산출물 현황. 파이프라인이 어디까지 진행됐는지 보여줍니다.
def count_lines(path):
    if not path.exists():
        return None
    return sum(1 for line in path.read_text(encoding='utf-8').splitlines() if line.strip())

print('데이터셋')
dataset = ROOT / 'data/processed/requirements_v0.3.0.jsonl'
print(f'  requirements_v0.3.0: {count_lines(dataset)}건'
      if dataset.exists() else '  없음 — scripts/data/build_dataset.py를 먼저 실행하세요')

print('\n앵커 풀 (few-shot 프롬프트에 주입되는 사례)')
for pool_path in sorted((ROOT / 'data/anchors').glob('anchor_pool_*.jsonl')):
    rows = [json.loads(l) for l in pool_path.read_text(encoding='utf-8').splitlines() if l.strip()]
    reviewed = sum(1 for r in rows if r.get('review_status') == '검토완료')
    print(f'  {pool_path.name}: 전체 {len(rows)}건 / 검토완료 {reviewed}건')
print('  * 전수 라벨링 기준 풀은 v2입니다. v3는 누적 앵커링 실험용으로 보존만 합니다.')

print('\nLLM 실행 결과')
runs_dir = ROOT / 'reports/current/claude_runs'
if runs_dir.exists():
    for run in sorted(runs_dir.iterdir()):
        results = run / 'results.jsonl'
        if results.exists():
            rows = [json.loads(l) for l in results.read_text(encoding='utf-8').splitlines() if l.strip()]
            ok = sum(1 for r in rows if r.get('status') == 'ok')
            print(f'  {run.name}: {ok}/{len(rows)}건 성공')
        elif (run / 'batch_info.json').exists():
            print(f'  {run.name}: 배치 제출됨 (--download로 결과 수신)')